# Train a VAE on Digits

*Hard · from [mlcode](https://mlcode.tech/problems/torch-vae-digits)*

Build a variational autoencoder, write its loss with `torch.distributions`,
and train it on handwritten digits.

The data is scikit-learn's 8x8 digits, flattened to 64 pixels and made
binary, so each pixel is `0` or `1`. A VAE has three parts:

- An **encoder** maps `x` to the mean `mu` and the log variance `logvar`
  of a Gaussian over a small latent vector `z`.
- A **reparameterised sample** `z = mu + exp(0.5 * logvar) * eps`, with
  `eps ~ N(0, I)`. The noise is outside the network, so gradients flow to
  `mu` and `logvar`.
- A **decoder** maps `z` to one logit for each pixel.

The loss is the negative ELBO, averaged over the batch:

```
recon = -sum over pixels of log Bernoulli(x | logits)
kl    =  sum over latent dims of KL( N(mu, sigma) || N(0, 1) )
loss  =  mean over the batch of (recon + kl)
```

`torch.distributions` gives you both terms: `Bernoulli(logits=...)` has a
`log_prob`, and `kl_divergence` knows the closed form for two `Normal`s.

Then write `train_vae(X)`, which builds a `VAE`, trains it with mini-batches
from a `DataLoader`, and returns it. This part is scored on the result.

### Function signature

```python
class VAE(nn.Module):
    def __init__(self, in_dim=64, latent_dim=8, hidden=128): ...
    def encode(self, x): ...      # -> (mu, logvar), each (n, latent_dim)
    def decode(self, z): ...      # -> logits, (n, in_dim)
    def forward(self, x): ...     # -> (logits, mu, logvar)


def negative_elbo(x, logits, mu, logvar):
def train_vae(X):
```

### Example

**Input**

```
x      = torch.tensor([[1.0, 0.0]])
logits = torch.tensor([[0.0, 0.0]])
mu     = torch.tensor([[1.0]])
logvar = torch.tensor([[0.0]])
negative_elbo(x, logits, mu, logvar)
```

**Output**

```
tensor(1.8863)
```

**Reasoning**

A logit of `0` gives each pixel probability `0.5`, so the reconstruction
term is `2 * log 2 = 1.3863`. The posterior is `N(1, 1)`, and its KL from
`N(0, 1)` is `0.5 * (1 + 1 - 1 - 0) = 0.5`. The sum is `1.8863`.

### Constraints and notes

- `negative_elbo` returns a scalar tensor that autograd can differentiate.
- `forward` must draw a new `z` with the reparameterisation trick. Do not
  call `.sample()`, which blocks the gradient. `rsample()` is fine.
- `train_vae` gets a float tensor of shape `(n, 64)` with values in
  `{0, 1}`. It must return a `VAE` with `latent_dim=8`.
- On held-out digits, the trained model must reach a negative ELBO below
  `25` nats, with a mean KL above `1.5` nats. A model that ignores its
  latent code cannot do this. An untrained model is near `44`.
- Keep training under about 10 seconds on a CPU.

---

Write your solution in the cell below, then run the grading cell at the bottom. Your work is recorded against the same account as the site, so a pass here shows there.

Keep the solution in the function you were given. The grading cell saves your definitions and the lines around them back to the site, not the whole session, so anything you type to poke about is left behind rather than ending up in your saved answer.

In [ ]:
#@title Connect this notebook to mlcode { display-mode: "form" }
#@markdown Paste the code from the problem page, then run this cell.
CODE = ""  #@param {type:"string"}

SITE = "https://mlcode.tech"
SLUG = "torch-vae-digits"

import json, urllib.request, urllib.error

def _mlc_get(path):
    with urllib.request.urlopen(SITE + path, timeout=30) as r:
        return r.read()

# The harness is fetched rather than pasted in, so the cases run here exactly
# as they would on the site.
for _mlc_n in ("harness.py", "notebook.py"):
    with open("/content/" + _mlc_n, "wb") as _mlc_fh:
        _mlc_fh.write(_mlc_get("/" + _mlc_n))

try:
    _mlc_meta = json.loads(_mlc_get(f"/api/colab?op=tests&code={CODE}&slug={SLUG}"))
    print("connected:", _mlc_meta["title"])
    print("torch:", __import__("torch").__version__)
except urllib.error.HTTPError as e:
    print("could not connect:", json.loads(e.read()).get("error", e.reason))
    print("Get a fresh code from the problem page; they last three hours.")


In [ ]:
import torch
from torch import nn
from torch.distributions import Bernoulli, Normal, kl_divergence


class VAE(nn.Module):
    """Gaussian encoder, Bernoulli decoder."""

    def __init__(self, in_dim=64, latent_dim=8, hidden=128):
        pass

    def encode(self, x):
        """x: (n, in_dim). Returns (mu, logvar), each (n, latent_dim)."""
        pass

    def decode(self, z):
        """z: (n, latent_dim). Returns logits of shape (n, in_dim)."""
        pass

    def forward(self, x):
        """Returns (logits, mu, logvar), with z drawn by reparameterisation."""
        pass


def negative_elbo(x, logits, mu, logvar):
    """Mean over the batch of the reconstruction term plus the KL term.

    returns: scalar torch.Tensor
    """
    pass


def train_vae(X):
    """Train a VAE(64, latent_dim=8) on X of shape (n, 64) and return it."""
    pass

In [ ]:
#@title Grade this notebook { display-mode: "form" }
#@markdown Runs the hidden tests and records the attempt on the site.
import json, sys, time, urllib.request, urllib.error

sys.path.insert(0, "/content")
import harness

# The tests are fetched now rather than stored in the notebook, so a reader
# cannot read them before attempting the problem.
_mlc_tests = json.loads(_mlc_get(f"/api/colab?op=tests&code={CODE}&slug={SLUG}"))["tests"]

# `In` is the execution history, not the current cells: a wrong first attempt,
# a cell that errored and three re-runs are all still in it. Saving that would
# put something on the site that does not even replay. Rebuild the source from
# what is actually defined now instead, and keep the history only as a fallback
# for a solution written at module level.
def _mlc_collect():
    import ast, inspect, types

    # This notebook's own form cells are not part of anyone's answer.
    _cells = [c for c in In[1:] if not c.lstrip().startswith("#@title")]

    _imports, _seen = [], set()
    _consts = dict()
    for _cell in _cells:
        for _line in _cell.split("\n"):
            _t = _line.strip()
            if (_t.startswith("import ") or _t.startswith("from ")) and _t not in _seen:
                _seen.add(_t)
                _imports.append(_t)
        try:
            _tree = ast.parse(_cell)
        except SyntaxError:
            continue
        # Module level constants the functions below may close over. Last
        # assignment wins, because an edited cell is run again.
        for _node in _tree.body:
            if isinstance(_node, ast.Assign) and len(_node.targets) == 1:
                _target = _node.targets[0]
                if isinstance(_target, ast.Name):
                    _text = ast.get_source_segment(_cell, _node)
                    if _text and len(_text) <= 300:
                        # A plain literal can go above the definitions, where a
                        # default argument can still see it. Anything else may
                        # call one of them, so it has to come after.
                        try:
                            ast.literal_eval(_node.value)
                            _lit = True
                        except Exception:
                            _lit = False
                        _consts[_target.id] = (_text, _lit)

    _defs, _defined = [], set()
    for _name, _obj in list(globals().items()):
        if _name.startswith("_mlc_") or _name in ("In", "Out"):
            continue
        if isinstance(_obj, (types.FunctionType, type)) and getattr(_obj, "__module__", "") == "__main__":
            try:
                _lead = inspect.getcomments(_obj) or ""
                _defs.append(_lead + inspect.getsource(_obj).rstrip())
                _defined.add(_name)
            except (OSError, TypeError):
                pass

    # Nothing defined means the solution is not shaped like a function, so fall
    # back to what was typed rather than returning an empty file.
    if not _defs:
        return "\n\n".join(_cells)

    _early, _late = [], []
    for _name, _pair in _consts.items():
        if _name in _defined or _name.startswith("_mlc_") or _name in ("CODE", "SITE", "SLUG"):
            continue
        # What is kept is the line of source, not the value it produced, so how
        # large the value is does not matter. The length bound above is what
        # keeps a pasted-in data literal out.
        (_early if _pair[1] else _late).append(_pair[0])

    _blocks = ["\n".join(_imports), "\n".join(_early),
               "\n\n\n".join(_defs), "\n".join(_late)]
    return "\n\n\n".join(b for b in _blocks if b.strip()) + "\n"

_mlc_source = _mlc_collect()

_mlc_started = time.time()
_mlc_results = harness.run_tests(_mlc_tests, dict(globals()), 60, _mlc_source, "submit")
_mlc_elapsed = int((time.time() - _mlc_started) * 1000)

_mlc_passed = sum(1 for r in _mlc_results if r["passed"])
for _mlc_r in _mlc_results:
    print(("  pass  " if _mlc_r["passed"] else "  FAIL  ") + _mlc_r["name"])
    if not _mlc_r["passed"] and _mlc_r["message"]:
        print("        " + _mlc_r["message"].replace("\n", "\n        "))
print(f"\n{_mlc_passed}/{len(_mlc_results)} passed")

_mlc_body = json.dumps({
    "code": CODE, "slug": SLUG, "passedCount": _mlc_passed,
    "totalCount": len(_mlc_results), "runtimeMs": _mlc_elapsed, "source": _mlc_source,
}).encode()
_mlc_req = urllib.request.Request(SITE + "/api/colab", data=_mlc_body,
                              headers={"Content-Type": "application/json"})
try:
    with urllib.request.urlopen(_mlc_req, timeout=30) as r:
        print("recorded on mlcode:", "solved" if json.load(r)["passed"] else "attempt saved")
except urllib.error.HTTPError as e:
    print("could not record:", json.loads(e.read()).get("error", e.reason))
